#**Person 1**

In [17]:
!pip install -q ultralytics supervision

In [18]:
import cv2
import json
import time
import numpy as np
import supervision as sv
from google.colab import files
from ultralytics import YOLO
from collections import defaultdict

In [19]:
model = YOLO("yolo26x.pt")

In [20]:
video_path = "/content/testVideo.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

print(f"Width  : {width}")
print(f"Height : {height}")
print(f"FPS    : {fps}")


Width  : 448
Height : 256
FPS    : 25.0


In [21]:
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    "tracked_output.mp4",
    fourcc,
    fps,
    (width, height)
)

In [22]:
# Configuration

PERSON_CLASS_ID = 0
CONFIDENCE_THRESHOLD = 0.4
TRACKER_TYPE = "botsort.yaml"
IOU_THRESHOLD = 0.5

ZONES = {
    "shelf_left_far":   (0.02, 0.02, 0.40, 0.32),
    "shelf_left_near":  (0.00, 0.32, 0.33, 0.75),
    "center_aisle":     (0.33, 0.05, 0.68, 0.78),
    "shelf_right_near": (0.65, 0.34, 1.00, 0.72),
    "shelf_right_far":  (0.58, 0.02, 1.00, 0.33),
    "entrance_area":    (0.00, 0.75, 1.00, 1.00),
}

ZONE_LABELS = {
    "shelf_left_far":"S1",
    "shelf_left_near":"S2",
    "center_aisle":"S3",
    "shelf_right_near":"S4",
    "shelf_right_far":"S5",
    "entrance_area":"EA"
}

MIN_DWELL_SECONDS = 2.0
QUEUE_ZONE = "center_aisle"
QUEUE_BUSY_THRESHOLD = 3
HEATMAP_GRID_SIZE = 20
HEATMAP_DECAY = 0.98

In [23]:
track_history = defaultdict(list)
id_map = {}
next_id = 1
customer_zone_state = {}
tracking_data = []
frame_count = 0

#**Person 2**

In [24]:
# Tracking Helper Functions

def get_color(display_id):
    np.random.seed(display_id)
    color = np.random.randint(50,255,size=3)
    return (int(color[0]),int(color[1]),int(color[2]))

def get_display_id(track_id):
    global next_id
    if track_id not in id_map:
        id_map[track_id] = next_id
        next_id += 1

    return id_map[track_id]

def get_center(x1,y1,x2,y2):
    return ((x1+x2)//2,(y1+y2)//2)

def draw_center(frame,center):
    cv2.circle(frame,center,3,(0,0,255),-1)

def update_track_history(display_id,center):
    track_history[display_id].append(center)
    if len(track_history[display_id])>30:
        track_history[display_id].pop(0)


def draw_box(frame,x1,y1,x2,y2,color,display_id):
    cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
    label = f"ID {display_id}"
    (tw,th),_ = cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.45,1)
    cv2.rectangle(frame,(x1,y1-th-6),(x1+tw+6,y1),color,-1)
    cv2.putText(frame,label,(x1+3,y1-3),cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,255),1,cv2.LINE_AA)

def draw_trajectory(frame,display_id,color):
    points = track_history[display_id]
    for i in range(1,len(points)):
        cv2.line(frame,points[i-1],points[i],color,2,cv2.LINE_AA)


#**Person 3**

In [25]:
# Zone Functions

def get_zone(x1, y1, x2, y2, frame_width, frame_height):
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2

    cx /= frame_width
    cy /= frame_height

    for zone_name, (zx1, zy1, zx2, zy2) in ZONES.items():
        if zx1 <= cx <= zx2 and zy1 <= cy <= zy2:
            return zone_name

    return None

def update_dwell(track_id,x1,y1,x2,y2,frame_width,frame_height,timestamp):
    zone = get_zone(x1,y1,x2,y2,frame_width,frame_height)
    state = customer_zone_state.setdefault(
        track_id,
        {
            "zone":None,
            "zone_entered_at":timestamp,
            "dwell_totals":{},
            "path":[]
        }
    )
    if zone != state["zone"]:
        state["zone"] = zone
        state["zone_entered_at"] = timestamp
        if zone is not None:
            if len(state["path"]) == 0 or state["path"][-1] != zone:
                state["path"].append(zone)
    else:
        elapsed = timestamp - state["zone_entered_at"]
        if zone is not None:
            state["dwell_totals"][zone] = (state["dwell_totals"].get(zone,0) + elapsed)

        state["zone_entered_at"] = timestamp

def get_customer_path(track_id):
    if track_id not in customer_zone_state:
        return []

    return customer_zone_state[track_id]["path"]

def get_all_customer_paths():
    return { tid:state["path"] for tid,state in customer_zone_state.items()}

def get_all_zone_averages():
    zone_totals = {}
    zone_counts = {}

    for state in customer_zone_state.values():
        for zone,seconds in state["dwell_totals"].items():
            if seconds < MIN_DWELL_SECONDS:
                continue
            zone_totals[zone] = zone_totals.get(zone,0)+seconds
            zone_counts[zone] = zone_counts.get(zone,0)+1
    averages = {}
    for zone in zone_totals:
        averages[zone] = round(zone_totals[zone]/zone_counts[zone],1)

    return averages

def get_common_flow_patterns(top_n=5):
    counts = {}
    for state in customer_zone_state.values():
        if len(state["path"]) == 0:
            continue
        p = " -> ".join(state["path"])
        counts[p] = counts.get(p,0)+1

    return sorted(counts.items(),key=lambda x:x[1],reverse=True)[:top_n]

In [26]:
frame_count = 0

while True:
    success, frame = cap.read()
    if not success:
        break

    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processing Frame {frame_count}")

    results = model.track(frame,persist=True,tracker=TRACKER_TYPE,classes=[PERSON_CLASS_ID],conf=CONFIDENCE_THRESHOLD,iou=IOU_THRESHOLD,verbose=False)
    boxes = results[0].boxes

    # Draw Store Zones
    for zone_name, (zx1, zy1, zx2, zy2) in ZONES.items():
        px1 = int(zx1 * width)
        py1 = int(zy1 * height)
        px2 = int(zx2 * width)
        py2 = int(zy2 * height)

        color = (0,255,0)
        if zone_name == QUEUE_ZONE:
            color = (0,255,255)

        cv2.rectangle(frame,(px1, py1),(px2, py2),color,2)
        cv2.putText(frame,ZONE_LABELS.get(zone_name, zone_name),(px1 + 5, py1 + 18),cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)
    if boxes.id is None:
        out.write(frame)
        continue

    frame_info = {
        "frame": frame_count,
        "detections": []
    }
    for box, track_id in zip(boxes.xyxy, boxes.id):
        x1, y1, x2, y2 = box.int().tolist()
        track_id = int(track_id)
        display_id = get_display_id(track_id)

        color = get_color(display_id)
        center = get_center(x1, y1, x2, y2)

        update_track_history(display_id, center)
        timestamp = frame_count / fps
        update_dwell(display_id,x1,y1,x2,y2,width,height,timestamp)
        zone = get_zone(x1,y1,x2,y2,width,height)

        draw_box(frame,x1,y1,x2,y2,color,display_id)
        draw_center(frame,center)
        draw_trajectory(frame,display_id,color)

        if zone is not None:
            cv2.putText(frame,ZONE_LABELS.get(zone, zone),(x1, y2 + 18),cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

        frame_info["detections"].append({
            "track_id": display_id,
            "bbox": [x1, y1, x2, y2],
            "zone": zone
        })
    tracking_data.append(frame_info)
    out.write(frame)

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 282ms
Prepared 1 package in 40ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Processing Frame 30
Processing Frame 60
Processing Frame 90
Processing Frame 120
Processing Frame 150
Processing Frame 180
Processing Frame 210
Processing Frame 240
Processing Frame 270
Processing Frame 300
Processing Frame 330
Processing Frame 360
Processing Frame 390
Processing Frame 420
Processing Frame 450
Processing Frame 480
Processing Frame 510
Processing Frame 540


#**Person 4**

In [27]:
# Analytics Variables
queue_state = {}
heatmap_grid = None
frame_crowd_counts = []

In [28]:

def init_heatmap():
    global heatmap_grid
    size = HEATMAP_GRID_SIZE
    heatmap_grid = [[0.0 for _ in range(size)] for _ in range(size)]

def update_queue(track_id,x1,y1,x2,y2,frame_width,frame_height,timestamp):
    zone = get_zone(x1,y1,x2,y2,frame_width,frame_height)
    if zone == QUEUE_ZONE:
        if track_id not in queue_state:
            queue_state[track_id] = {"entered_at": timestamp}
    else:
        queue_state.pop(track_id, None)

def get_queue_count():
    return len(queue_state)

def get_queue_waiting_times(current_timestamp):
    waits = {}
    for track_id, state in queue_state.items():
        waits[track_id] = round(current_timestamp -state["entered_at"],1)

    return waits

def get_average_waiting_time(current_timestamp):
    waits = get_queue_waiting_times(current_timestamp)
    if len(waits) == 0:
        return 0.0

    return round(sum(waits.values()) /len(waits),1)

def is_queue_busy():
    return get_queue_count() >= QUEUE_BUSY_THRESHOLD

def update_heatmap(x1,y1,x2,y2,frame_width,frame_height):
    global heatmap_grid
    if heatmap_grid is None:
        init_heatmap()

    size = HEATMAP_GRID_SIZE
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    col = min(int((cx / frame_width) * size),size - 1)
    row = min(int((cy / frame_height) * size),size - 1)
    for dr in (-1,0,1):
        for dc in (-1,0,1):
            rr = row + dr
            cc = col + dc
            if 0 <= rr < size and 0 <= cc < size:
                if dr == 0 and dc == 0:
                    heatmap_grid[rr][cc] += 1.0
                else:
                    heatmap_grid[rr][cc] += 0.35

def get_heatmap():
    if heatmap_grid is None:
        init_heatmap()
    return heatmap_grid

def record_frame_crowd_count(track_ids):
    frame_crowd_counts.append(len(track_ids))

def get_current_crowd_count(track_ids):
    return len(track_ids)

def get_crowd_density_stats():
    if len(frame_crowd_counts) == 0:
        return {"average":0,"max":0,"min":0}

    return {
        "average": round(sum(frame_crowd_counts)/ len(frame_crowd_counts),1),
        "max": max(frame_crowd_counts),
        "min": min(frame_crowd_counts)
    }

def get_summary_statistics(current_timestamp):
    return {
        "queue_count":get_queue_count(),
        "average_waiting_time":get_average_waiting_time(current_timestamp),
        "queue_busy":is_queue_busy(),
        "crowd_density":get_crowd_density_stats()
    }

#**Person 5**

In [29]:
# Evaluation Variables
track_frame_counts = {}
total_frames_processed = 0
frame_times = []
timer_start = None

In [30]:
# Evaluation functions

def start_timer():
    global timer_start
    timer_start = time.time()

def stop_timer():
    global timer_start
    if timer_start is None:
        return None

    elapsed = time.time() - timer_start
    frame_times.append(elapsed)
    timer_start = None
    return elapsed

def record_frame(tracked):
    global total_frames_processed
    total_frames_processed += 1
    for track_id, *_ in tracked:
        track_frame_counts[track_id] = (track_frame_counts.get(track_id, 0) + 1)

def get_track_lengths():
    return dict(track_frame_counts)

def get_total_unique_ids():
    return len(track_frame_counts)

def get_average_track_length():
    if not track_frame_counts:
        return 0.0

    return round(sum(track_frame_counts.values())/ len(track_frame_counts),1)

def get_fragmented_tracks(min_frames=5):
    return {
      track_id: length
      for track_id, length in track_frame_counts.items()
      if length < min_frames
    }

def get_fragmentation_rate(min_frames=5):
    if not track_frame_counts:
        return 0.0

    fragmented = get_fragmented_tracks(min_frames)
    return round(len(fragmented)/ len(track_frame_counts)* 100,1)

def get_average_fps():
    if not frame_times:
        return 0.0

    avg_time = sum(frame_times) / len(frame_times)
    if avg_time == 0:
        return 0.0

    return round(1 / avg_time,1)

def get_average_frame_time_ms():
    if not frame_times:
        return 0.0

    return round((sum(frame_times) / len(frame_times))* 1000,1)

def get_evaluation_report():
    return {
        "total_frames_processed":total_frames_processed,
        "total_unique_ids":get_total_unique_ids(),
        "average_track_length_frames":get_average_track_length(),
        "fragmented_tracks_count":len(get_fragmented_tracks()),
        "fragmentation_rate_percent":get_fragmentation_rate(),
        "average_fps":get_average_fps(),
        "average_frame_time_ms":get_average_frame_time_ms()
    }

In [31]:
frame_count = 0
id_map = {}
next_id = 1

while True:
    success, frame = cap.read()
    if not success:
        break

    start_timer()
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processing Frame {frame_count}")

    results = model.track(
        frame,
        persist=True,
        tracker=TRACKER_TYPE,
        classes=[PERSON_CLASS_ID],
        conf=CONFIDENCE_THRESHOLD,
        iou=IOU_THRESHOLD,
        verbose=False
    )

    boxes = results[0].boxes

    # Draw Store Zones
    for zone_name, (zx1, zy1, zx2, zy2) in ZONES.items():
        px1 = int(zx1 * width)
        py1 = int(zy1 * height)
        px2 = int(zx2 * width)
        py2 = int(zy2 * height)

        color = (0, 255, 0)
        if zone_name == QUEUE_ZONE:
            color = (0, 255, 255)

        cv2.rectangle(frame,(px1, py1),(px2, py2),color,2)
        cv2.putText( frame, ZONE_LABELS.get(zone_name, zone_name), (px1 + 5, py1 + 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    if boxes.id is None:
        stop_timer()
        out.write(frame)
        continue

    frame_info = {
        "frame": frame_count,
        "detections": []
    }

    tracked = []
    track_ids_this_frame = []
    timestamp = frame_count / fps

    # Process Every Detection
    for box, track_id in zip(boxes.xyxy, boxes.id):
        x1, y1, x2, y2 = box.int().tolist()
        track_id = int(track_id)
        display_id = get_display_id(track_id)

        color = get_color(display_id)
        center = get_center(x1,y1,x2,y2)

        update_track_history(display_id,center)
        zone = get_zone(x1,y1,x2,y2,width,height)
        update_dwell(display_id,x1,y1,x2,y2,width,height,timestamp)
        update_queue(display_id,x1,y1,x2,y2,width,height,timestamp)
        update_heatmap(x1,y1,x2,y2,width,height)

        draw_box(frame,x1,y1,x2,y2,color,display_id)
        draw_center(frame,center)
        draw_trajectory(frame,display_id,color)
        if zone is not None:
            cv2.putText(frame,ZONE_LABELS.get(zone, zone),(x1, y2 + 18),cv2.FONT_HERSHEY_SIMPLEX,0.5,color,2)

        frame_info["detections"].append(
            {
                "track_id": display_id,
                "bbox": [x1,y1,x2,y2],
                "zone": zone
            }
        )
        tracked.append((display_id,x1,y1,x2,y2))

        track_ids_this_frame.append(  display_id)
    # Save Frame Information
    tracking_data.append(frame_info)
    record_frame(tracked)
    record_frame_crowd_count(track_ids_this_frame)

    # Debug
    print("Frames:",total_frames_processed)
    print("Tracks:",len(track_frame_counts))
    print("Crowd Frames:",len(frame_crowd_counts))
    stop_timer()
    out.write(frame)

In [32]:
cap.release()
out.release()

print("Tracking Finished Successfully!")
print("\n==============================")
print("ZONE ANALYTICS")
print("==============================")

print("\nAverage Dwell Time")
print(get_all_zone_averages())

print("\nCustomer Paths")
print(get_all_customer_paths())

print("\nCommon Flow Patterns")
print(get_common_flow_patterns())

print("\n==============================")
print("QUEUE ANALYTICS")
print("==============================")

print(get_summary_statistics(frame_count / fps))

print("\nCrowd Density")
print(get_crowd_density_stats())

print("\n==============================")
print("EVALUATION")
print("==============================")

print(get_evaluation_report())

with open("tracking_data.json", "w") as f:
    json.dump(tracking_data,f,indent=4)

from google.colab import files
files.download("tracked_output.mp4")
files.download("tracking_data.json")

Tracking Finished Successfully!

ZONE ANALYTICS

Average Dwell Time
{'center_aisle': 8.7, 'shelf_left_far': 3.9, 'entrance_area': 2.5, 'shelf_left_near': 2.5, 'shelf_right_far': 2.1}

Customer Paths
{1: ['center_aisle'], 2: ['center_aisle', 'shelf_left_far'], 3: ['center_aisle'], 4: ['entrance_area', 'center_aisle', 'entrance_area', 'center_aisle', 'shelf_right_near', 'center_aisle', 'shelf_right_near', 'center_aisle'], 5: ['center_aisle', 'shelf_right_near'], 6: ['entrance_area'], 7: ['shelf_left_far'], 8: ['shelf_right_far'], 9: ['center_aisle'], 10: ['shelf_left_far', 'center_aisle', 'shelf_left_far', 'center_aisle'], 11: ['center_aisle', 'shelf_left_near'], 12: ['shelf_left_far'], 13: ['center_aisle'], 14: ['center_aisle'], 15: ['shelf_right_far'], 16: ['center_aisle'], 17: ['entrance_area', 'center_aisle'], 18: ['center_aisle', 'shelf_left_near', 'entrance_area'], 19: ['center_aisle'], 20: ['center_aisle'], 21: ['center_aisle'], 22: ['shelf_right_far'], 23: ['entrance_area'], 24: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
!pip install -q gradio

In [ ]:
import gradio as gr
import cv2
import json
import tempfile
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# NOTE: uses the `model` object already created in your earlier YOLO cell.
# Do NOT reload it here — reloading a large model on every run is what was
# causing slow / failed runs. If `model` isn't defined yet, run that cell first.
assert "model" in globals(), "Run the `model = YOLO(...)` cell above first."

DEFAULT_CONF = CONFIDENCE_THRESHOLD
DEFAULT_IOU = IOU_THRESHOLD
DEFAULT_MIN_DWELL = MIN_DWELL_SECONDS
DEFAULT_QUEUE_BUSY = QUEUE_BUSY_THRESHOLD

ACCENT_TEAL = "#2FD9C4"
ACCENT_AMBER = "#F2B84B"
ACCENT_TEAL_BGR = (196, 217, 47)
ACCENT_AMBER_BGR = (75, 184, 242)

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@400;500;600;700;800&family=Comfortaa:wght@500;600;700&family=JetBrains+Mono:wght@400;500;600&display=swap');
@import url('https://fonts.googleapis.com/css2?family=Material+Symbols+Outlined:opsz,wght,FILL,GRAD@20..48,300..500,0,0&display=swap');

.material-symbols-outlined{
    font-family:'Material Symbols Outlined'!important;font-weight:400;font-style:normal;
    line-height:1;letter-spacing:normal;text-transform:none;white-space:nowrap;
    vertical-align:middle;font-size:20px;
}
.gradio-container{
    background:#0D1117!important;font-family:'Montserrat',sans-serif!important;
    max-width:1360px!important;padding:28px 32px 48px 32px!important;
}
body,p,span,div,label{ font-family:'Montserrat',sans-serif; }

#hero{
    position:relative; border:1px solid #263041; border-radius:18px;
    padding:44px 44px 38px 44px;
    background:
        radial-gradient(circle at 12% 20%, rgba(47,217,196,0.10), transparent 45%),
        radial-gradient(circle at 88% 0%, rgba(242,184,75,0.08), transparent 40%),
        linear-gradient(180deg,#121821 0%,#0D1117 100%);
    overflow:hidden; margin-bottom:30px;
}
#hero::before{
    content:""; position:absolute; left:0; right:0; top:0; height:2px;
    background:linear-gradient(90deg,transparent,#2FD9C4,transparent);
    animation:scan 5s linear infinite; opacity:.7;
}
@keyframes scan{0%{transform:translateY(0);}100%{transform:translateY(230px);}}
.hud-corner{position:absolute;width:18px;height:18px;border-color:#2FD9C4;opacity:.55;}
.hud-tl{top:12px;left:12px;border-top:2px solid;border-left:2px solid;}
.hud-tr{top:12px;right:12px;border-top:2px solid;border-right:2px solid;}
.hud-bl{bottom:12px;left:12px;border-bottom:2px solid;border-left:2px solid;}
.hud-br{bottom:12px;right:12px;border-bottom:2px solid;border-right:2px solid;}
.eyebrow{
    font-family:'Comfortaa',sans-serif;font-weight:600;letter-spacing:.12em;
    font-size:12.5px;color:#2FD9C4;text-transform:uppercase;margin-bottom:14px;
    display:flex;align-items:center;gap:8px;
}
.main-header{
    font-family:'Montserrat',sans-serif;font-size:2.15rem;font-weight:800;
    color:#F5F7F9;margin:0 0 12px 0;letter-spacing:-.015em;line-height:1.3;
    display:flex;align-items:center;gap:12px;
}
.main-header .material-symbols-outlined{font-size:30px;color:#2FD9C4;}
.sub-header{
    font-family:'Comfortaa',sans-serif;font-weight:500;font-size:14.5px;
    color:#9BA6B3;margin:0;max-width:700px;line-height:1.8;
}
.pipeline-rail{ display:flex; gap:10px; margin:22px 0 6px 0; flex-wrap:wrap; }
.stage-chip{
    display:flex;align-items:center;gap:6px;font-family:'Montserrat',sans-serif;
    font-weight:600;font-size:12px;padding:9px 15px;border-radius:999px;
    border:1px solid #2FD9C4;color:#7CF0DF;background:rgba(47,217,196,0.08);
    white-space:nowrap;
}
.stage-chip .material-symbols-outlined{font-size:15px;}
.stage-chip .n{font-family:'JetBrains Mono',monospace;color:#5EE8D3;margin-right:2px;font-weight:400;}

.section-title{
    display:flex;align-items:center;gap:10px;font-size:1.08rem;font-weight:700;
    color:#EDF1F4;margin-top:22px;margin-bottom:16px;padding:8px 0 8px 16px;
    border-left:4px solid #2FD9C4;
}
.section-title .material-symbols-outlined{
    font-size:19px;color:#2FD9C4;background:rgba(47,217,196,0.12);
    border-radius:7px;padding:4px;
}
.stat-row{ display:flex; gap:16px; flex-wrap:wrap; margin-bottom:8px; }
.stat-card{
    flex:1; min-width:160px; background:#151A21; border:1px solid #262E3A;
    border-radius:12px; padding:20px 22px; transition:border-color .2s ease;
}
.stat-card:hover{ border-color:#364154; }
.stat-label{
    font-weight:700;color:#8FA3B0;font-size:11.5px;text-transform:uppercase;
    letter-spacing:.06em; margin-bottom:8px;
}
.stat-value{
    color:#E8ECEF;font-size:1.8rem;font-family:'JetBrains Mono',monospace;font-weight:600;
}
.themed-table{
    width:100%;border-collapse:collapse;background-color:#151A21;
    border:1px solid #262E3A;border-radius:10px;overflow:hidden;margin-bottom:10px;
}
.themed-table th{ text-align:left;padding:11px 16px;color:#8FA3B0;font-size:12px;background:#1B222C; }
.themed-table td{ padding:9px 16px;border-bottom:1px solid #262E3A;color:#E8ECEF; }
.themed-table td.mono{ color:#8B94A3;font-family:'JetBrains Mono',monospace; }
.flow-list{ padding-left:20px; margin:0; }
.flow-list li{ margin-bottom:9px; color:#E8ECEF; }
.flow-list .count{ color:#8B94A3; }
.report-card{
    background-color:#151A21;border:1px solid #262E3A;border-left:4px solid var(--badge,#2FD9C4);
    border-radius:10px;padding:20px 24px;color:#B7C1CC;line-height:1.8;
}
.report-card b{ color:#E8ECEF; }
label span, .gr-form label{ color:#C4CDD6!important;font-weight:600!important;font-size:13.5px!important; }
input, textarea, select{ background:#10151C!important;color:#E8ECEF!important;border-color:#2A323F!important; }
.gr-button, button{ font-family:'Montserrat',sans-serif!important; font-weight:700!important; border-radius:9px!important; }
button.primary, .gr-button-primary{ background:#2FD9C4!important; color:#0D1117!important; border:none!important; }
button.secondary, .gr-button-secondary{ background:#10151C!important; color:#E8ECEF!important; border:1px solid #2A323F!important; }
input[type=range]{ accent-color:#2FD9C4!important; }
.tabs > .tab-nav button{ font-family:'Montserrat',sans-serif!important;font-weight:600!important; color:#8B94A3!important; }
.tabs > .tab-nav button.selected{ color:#2FD9C4!important; }
.gr-accordion, .label-wrap{ border-color:#262E3A!important; background:#10151C!important; }
.footer-strip{
    text-align:center;color:#4A5568;font-family:'JetBrains Mono',monospace;
    font-size:11px;margin-top:36px;letter-spacing:.05em;
}
"""

PIPELINE_STAGES = [
    {"n": "01", "name": "Person Detection",     "icon": "person_search"},
    {"n": "02", "name": "Multi-Object Tracking", "icon": "my_location"},
    {"n": "03", "name": "Zones & Dwell Time",    "icon": "map"},
    {"n": "04", "name": "Retail Analytics",      "icon": "monitoring"},
    {"n": "05", "name": "Model Evaluation",      "icon": "speed"},
    {"n": "06", "name": "Dashboard",             "icon": "dashboard"},
]


def icon(name, size=None):
    style = f' style="font-size:{size}px;"' if size else ""
    return f'<span class="material-symbols-outlined"{style}>{name}</span>'


def section_title(name, text):
    return f'<p class="section-title">{icon(name)}{text}</p>'


def stat_card(label, value):
    return f'<div class="stat-card"><div class="stat-label">{label}</div><div class="stat-value">{value}</div></div>'


def stat_row(*cards):
    return f'<div class="stat-row">{"".join(cards)}</div>'


def pipeline_rail_html():
    chips = "".join(
        f'<div class="stage-chip">{icon(s["icon"], size=15)}<span class="n">{s["n"]}</span>{s["name"]}</div>'
        for s in PIPELINE_STAGES
    )
    return f'<div class="pipeline-rail">{chips}</div>'


HERO_HTML = f"""
<div id="hero">
    <div class="hud-corner hud-tl"></div><div class="hud-corner hud-tr"></div>
    <div class="hud-corner hud-bl"></div><div class="hud-corner hud-br"></div>
    <div class="eyebrow">{icon('bolt')}SMART RETAIL ANALYTICS — LIVE DASHBOARD</div>
    <p class="main-header">{icon('storefront')}Customer tracking, queues & store analytics — end to end.</p>
    <p class="sub-header">Full pipeline: detection → tracking → zones → analytics → evaluation.
    Upload a store camera clip below to run it.</p>
</div>
{pipeline_rail_html()}
"""


def _pretty_zone(zone_name):
    return zone_name.replace("_", " ").title() if zone_name else "Unknown"


def zones_table_html(dwell_averages):
    if not dwell_averages:
        return '<p style="color:#8B94A3;">No zone dwell data yet.</p>'
    rows = "".join(
        f'<tr><td>{_pretty_zone(z)}</td><td class="mono">{s}s</td></tr>'
        for z, s in sorted(dwell_averages.items(), key=lambda i: -i[1])
    )
    return f'<table class="themed-table"><tr><th>ZONE</th><th>AVG. DWELL TIME</th></tr>{rows}</table>'


def flow_list_html(flow_patterns):
    if not flow_patterns:
        return '<p style="color:#8B94A3;">No flow pattern data yet.</p>'
    items = "".join(
        f'<li>{" → ".join(_pretty_zone(z) for z in p.split(" -> "))} '
        f'<span class="count">— {c} customer{"s" if c != 1 else ""}</span></li>'
        for p, c in flow_patterns
    )
    return f'<ul class="flow-list">{items}</ul>'


def report_card_html(report):
    frag_rate = report["fragmentation_rate_percent"]
    if frag_rate < 10:
        label, badge, note = "Excellent", ACCENT_TEAL, "The system almost never loses track of a customer."
    elif frag_rate < 25:
        label, badge = "Good", ACCENT_AMBER
        note = ("A few customers were briefly re-tracked as \"new\" people, usually because someone "
                "else walked in front of them for a moment.")
    else:
        label, badge = "Needs improvement", "#FF6B6B"
        note = "Quite a few customers got split into multiple short IDs — likely from crowding or occlusion."

    return f"""
<div class="report-card" style="--badge:{badge};">
<b>Tracking quality: {label}</b><br>
<span style="color:#8B94A3;">{note}</span><br><br>
Processed <b>{report['total_frames_processed']} frames</b> at an average of
<b>{report['average_fps']} frames per second</b>.<br>
Followed <b>{report['total_unique_ids']} different people</b> through the store.<br>
On average, each person stayed visible for <b>{report['average_track_length_frames']} frames</b>
before the system lost or re-assigned their ID.<br>
<b>{report['fragmented_tracks_count']}</b> of those {report['total_unique_ids']} people
({frag_rate}%) were only tracked very briefly.
</div>
"""


# ============================================================
# PIPELINE — plain function, runs fully, returns final results.
# No live per-frame streaming (that was the source of the errors).
# ============================================================
def run_pipeline(video_file, confidence, iou, min_dwell, queue_busy_threshold, progress=gr.Progress()):
    if video_file is None:
        raise gr.Error("Please upload a video first.")

    track_history = defaultdict(list)
    id_map, next_id_holder = {}, {"v": 1}
    customer_zone_state, queue_state = {}, {}
    tracking_data = []
    heatmap_grid = [[0.0 for _ in range(HEATMAP_GRID_SIZE)] for _ in range(HEATMAP_GRID_SIZE)]
    frame_crowd_counts = []
    track_frame_counts = {}
    frame_times = []

    def get_display_id(track_id):
        if track_id not in id_map:
            id_map[track_id] = next_id_holder["v"]
            next_id_holder["v"] += 1
        return id_map[track_id]

    def get_center(x1, y1, x2, y2):
        return (x1 + x2) // 2, (y1 + y2) // 2

    def update_track_history(display_id, center):
        track_history[display_id].append(center)
        if len(track_history[display_id]) > 30:
            track_history[display_id].pop(0)

    def draw_box(frame, x1, y1, x2, y2, display_id, zone):
        label = f"ID {display_id} | {ZONE_LABELS.get(zone, '?')}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), ACCENT_TEAL_BGR, 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        cv2.rectangle(frame, (x1, max(y1 - th - 10, 0)), (x1 + tw + 8, y1), ACCENT_TEAL_BGR, -1)
        cv2.putText(frame, label, (x1 + 4, max(y1 - 6, th)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (12, 17, 22), 2)

    def draw_trajectory(frame, display_id):
        points = track_history[display_id]
        for i in range(1, len(points)):
            cv2.line(frame, points[i - 1], points[i], ACCENT_AMBER_BGR, 2, cv2.LINE_AA)

    def draw_zones(frame, w, h):
        for zone_name, (zx1, zy1, zx2, zy2) in ZONES.items():
            px1, py1, px2, py2 = int(zx1 * w), int(zy1 * h), int(zx2 * w), int(zy2 * h)
            color = ACCENT_AMBER_BGR if zone_name == QUEUE_ZONE else ACCENT_TEAL_BGR
            cv2.rectangle(frame, (px1, py1), (px2, py2), color, 2)
            cv2.putText(frame, ZONE_LABELS.get(zone_name, zone_name), (px1 + 5, py1 + 18),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    def get_zone(x1, y1, x2, y2, w, h):
        cx, cy = ((x1 + x2) / 2) / w, ((y1 + y2) / 2) / h
        for zone_name, (zx1, zy1, zx2, zy2) in ZONES.items():
            if zx1 <= cx <= zx2 and zy1 <= cy <= zy2:
                return zone_name
        return None

    def update_dwell(track_id, x1, y1, x2, y2, w, h, ts):
        zone = get_zone(x1, y1, x2, y2, w, h)
        state = customer_zone_state.setdefault(
            track_id, {"zone": None, "zone_entered_at": ts, "dwell_totals": {}, "path": []}
        )
        if zone != state["zone"]:
            state["zone"] = zone
            state["zone_entered_at"] = ts
            if zone is not None and (not state["path"] or state["path"][-1] != zone):
                state["path"].append(zone)
        else:
            elapsed = ts - state["zone_entered_at"]
            if zone is not None:
                state["dwell_totals"][zone] = state["dwell_totals"].get(zone, 0) + elapsed
            state["zone_entered_at"] = ts
        return zone

    def update_queue(track_id, zone, ts):
        if zone == QUEUE_ZONE:
            queue_state.setdefault(track_id, {"entered_at": ts})
        else:
            queue_state.pop(track_id, None)

    def update_heatmap(x1, y1, x2, y2, w, h):
        size = HEATMAP_GRID_SIZE
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        col = min(int((cx / w) * size), size - 1)
        row = min(int((cy / h) * size), size - 1)
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                rr, cc = row + dr, col + dc
                if 0 <= rr < size and 0 <= cc < size:
                    heatmap_grid[rr][cc] += 1.0 if (dr == 0 and dc == 0) else 0.35

    def get_zone_averages():
        totals, counts = {}, {}
        for st in customer_zone_state.values():
            for z, secs in st["dwell_totals"].items():
                if secs < min_dwell:
                    continue
                totals[z] = totals.get(z, 0) + secs
                counts[z] = counts.get(z, 0) + 1
        return {z: round(totals[z] / counts[z], 1) for z in totals}

    def get_flow_patterns(top_n=5):
        counts = {}
        for st in customer_zone_state.values():
            if not st["path"]:
                continue
            p = " -> ".join(st["path"])
            counts[p] = counts.get(p, 0) + 1
        return sorted(counts.items(), key=lambda x: x[1], reverse=True)[:top_n]

    def get_queue_stats(ts):
        waits = {tid: round(ts - s["entered_at"], 1) for tid, s in queue_state.items()}
        avg_wait = round(sum(waits.values()) / len(waits), 1) if waits else 0.0
        return len(queue_state), avg_wait, len(queue_state) >= queue_busy_threshold

    def get_crowd_stats():
        if not frame_crowd_counts:
            return {"average": 0}
        return {"average": round(sum(frame_crowd_counts) / len(frame_crowd_counts), 1)}

    def get_evaluation():
        if not track_frame_counts:
            avg_len, frag_count, frag_rate = 0.0, 0, 0.0
        else:
            avg_len = round(sum(track_frame_counts.values()) / len(track_frame_counts), 1)
            fragmented = {k: v for k, v in track_frame_counts.items() if v < 5}
            frag_count = len(fragmented)
            frag_rate = round(frag_count / len(track_frame_counts) * 100, 1)
        avg_fps = round(1 / (sum(frame_times) / len(frame_times)), 1) if frame_times else 0.0
        return {
            "total_frames_processed": len(frame_times),
            "total_unique_ids": len(track_frame_counts),
            "average_track_length_frames": avg_len,
            "fragmented_tracks_count": frag_count,
            "fragmentation_rate_percent": frag_rate,
            "average_fps": avg_fps,
        }

    cap = cv2.VideoCapture(video_file)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1

    out_path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    frame_count = 0
    while True:
        success, frame = cap.read()
        if not success:
            break
        frame_count += 1
        t0 = time.time()
        ts = frame_count / fps

        results = model.track(frame, persist=True, tracker=TRACKER_TYPE,
                               classes=[PERSON_CLASS_ID], conf=confidence, iou=iou, verbose=False)
        boxes = results[0].boxes
        draw_zones(frame, width, height)

        track_ids_this_frame = []
        frame_info = {"frame": frame_count, "detections": []}

        if boxes.id is not None:
            for box, track_id in zip(boxes.xyxy, boxes.id):
                x1, y1, x2, y2 = box.int().tolist()
                track_id = int(track_id)
                display_id = get_display_id(track_id)
                center = get_center(x1, y1, x2, y2)
                update_track_history(display_id, center)

                zone = update_dwell(display_id, x1, y1, x2, y2, width, height, ts)
                update_queue(display_id, zone, ts)
                update_heatmap(x1, y1, x2, y2, width, height)

                draw_box(frame, x1, y1, x2, y2, display_id, zone)
                draw_trajectory(frame, display_id)

                frame_info["detections"].append({"track_id": display_id, "bbox": [x1, y1, x2, y2], "zone": zone})
                track_frame_counts[display_id] = track_frame_counts.get(display_id, 0) + 1
                track_ids_this_frame.append(display_id)

            tracking_data.append(frame_info)
            frame_crowd_counts.append(len(track_ids_this_frame))

        frame_times.append(time.time() - t0)
        out.write(frame)
        progress(frame_count / total_frames, desc=f"Frame {frame_count}/{total_frames}")

    cap.release()
    out.release()

    json_path = tempfile.NamedTemporaryFile(suffix=".json", delete=False).name
    with open(json_path, "w") as f:
        json.dump(tracking_data, f, indent=4)

    heatmap_path = tempfile.NamedTemporaryFile(suffix=".png", delete=False).name
    grid = np.array(heatmap_grid, dtype=np.float32)
    if grid.max() > 0:
        grid = (grid / grid.max()) ** 0.45
    plt.figure(figsize=(6, 6), facecolor="#0D1117")
    ax = plt.gca()
    ax.set_facecolor("#0D1117")
    plt.imshow(grid, cmap="turbo", interpolation="bicubic")
    plt.title("Customer Movement Heatmap", color="#E8ECEF")
    plt.axis("off")
    cb = plt.colorbar(fraction=0.046, pad=0.04)
    cb.ax.yaxis.set_tick_params(color="#8B94A3")
    plt.setp(plt.getp(cb.ax.axes, "yticklabels"), color="#8B94A3")
    plt.savefig(heatmap_path, bbox_inches="tight", dpi=150, facecolor="#0D1117")
    plt.close()

    evald = get_evaluation()
    q_count, q_wait, q_busy = get_queue_stats(frame_count / fps)
    crowd = get_crowd_stats()

    live_stats_html = stat_row(
        stat_card("Queue count", q_count),
        stat_card("Avg waiting (s)", q_wait),
        stat_card("Crowd (avg)", crowd["average"]),
        stat_card("Status", "Busy" if q_busy else "Normal"),
    )
    perf_stats_html = stat_row(
        stat_card("Processing speed", f"{evald['average_fps']} fps"),
        stat_card("People counted", evald["total_unique_ids"]),
        stat_card("Tracking mix-ups", f"{evald['fragmentation_rate_percent']}%"),
        stat_card("Avg. time tracked", f"{evald['average_track_length_frames']} frm"),
    )

    return (
        out_path,                              # video_output
        heatmap_path,                          # heatmap_output
        live_stats_html,                       # live_stats
        zones_table_html(get_zone_averages()), # zones_table
        flow_list_html(get_flow_patterns()),   # flow_list
        perf_stats_html,                       # perf_stats
        report_card_html(evald),               # perf_report
        json_path,                             # json_output
    )


# ============================================================
# LAYOUT
# ============================================================
with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Base(), title="Smart Retail Analytics") as demo:
    gr.HTML(HERO_HTML)

    with gr.Row():
        with gr.Column(scale=1, min_width=320):
            gr.HTML(section_title("videocam", "Video Source"))
            video_input = gr.Video(label="Upload store video", sources=["upload"])

            gr.HTML(section_title("tune", "Settings"))
            confidence_slider = gr.Slider(0.1, 0.9, value=DEFAULT_CONF, step=0.05, label="Detection confidence")
            iou_slider = gr.Slider(0.1, 0.9, value=DEFAULT_IOU, step=0.05, label="IOU threshold")
            min_dwell_slider = gr.Slider(0.5, 10.0, value=DEFAULT_MIN_DWELL, step=0.5, label="Min dwell time (s)")
            queue_busy_slider = gr.Slider(1, 10, value=DEFAULT_QUEUE_BUSY, step=1, label="Queue busy threshold")

            with gr.Accordion("Current zone / queue config", open=False):
                gr.Markdown(
                    f"- Queue zone: `{QUEUE_ZONE}`\n"
                    f"- Heatmap grid: `{HEATMAP_GRID_SIZE}x{HEATMAP_GRID_SIZE}`\n"
                    f"- Zones defined: `{', '.join(ZONE_LABELS.values())}`"
                )

            run_btn = gr.Button("▶  Run analysis", variant="primary")

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("Live Feed"):
                    gr.HTML(section_title("query_stats", "Statistics"))
                    live_stats = gr.HTML(stat_row(
                        stat_card("Queue count", "—"), stat_card("Avg waiting (s)", "—"),
                        stat_card("Crowd (avg)", "—"), stat_card("Status", "—"),
                    ))
                    gr.HTML(section_title("smart_display", "Annotated video"))
                    video_output = gr.Video(label="Tracked output", show_label=False)

                with gr.Tab("Heatmap & Zones"):
                    with gr.Row():
                        with gr.Column():
                            gr.HTML(section_title("map", "Store heatmap"))
                            heatmap_output = gr.Image(label="Heatmap", show_label=False)
                        with gr.Column():
                            gr.HTML(section_title("schedule", "Average dwell time per zone (s)"))
                            zones_table = gr.HTML(zones_table_html({}))
                            gr.HTML(section_title("alt_route", "Most common customer flow patterns"))
                            flow_list = gr.HTML(flow_list_html([]))

                with gr.Tab("Performance"):
                    gr.HTML(section_title("speed", "Tracking quality"))
                    perf_stats = gr.HTML(stat_row(
                        stat_card("Processing speed", "—"), stat_card("People counted", "—"),
                        stat_card("Tracking mix-ups", "—"), stat_card("Avg. time tracked", "—"),
                    ))
                    gr.HTML(section_title("summarize", "Full report"))
                    perf_report = gr.HTML(report_card_html(
                        {"fragmentation_rate_percent": 0, "total_frames_processed": 0, "average_fps": 0,
                         "total_unique_ids": 0, "average_track_length_frames": 0, "fragmented_tracks_count": 0}
                    ))

            gr.HTML(section_title("download", "Export"))
            json_output = gr.File(label="Tracking data (JSON)")

    gr.HTML('<div class="footer-strip">SMART RETAIL ANALYTICS · NTI TEAM PROJECT · BUILT WITH GRADIO</div>')

    run_btn.click(
        fn=run_pipeline,
        inputs=[video_input, confidence_slider, iou_slider, min_dwell_slider, queue_busy_slider],
        outputs=[video_output, heatmap_output, live_stats, zones_table, flow_list, perf_stats, perf_report, json_output],
    )

demo.queue().launch(share=True, debug=True)

/tmp/ipykernel_607/3620242403.py:482: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Base(), title="Smart Retail Analytics") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fe0650d3dfbedd94cd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/components/video.py:303: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
